# MLP Training and Experiments for MAGIC Telescope

Training and evaluating neural network models on the MAGIC Gamma Telescope dataset using our custom numpy-based framework.

## 1. Setup and Data Loading

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from src.preprocessing import load_magic_dataset, preprocess_data, create_batches
from src.utils import set_seed, print_model_summary

set_seed(42)

# Load data
X, y = load_magic_dataset()
print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"Class distribution: {np.bincount(y.astype(int))}")

# Preprocess
data = preprocess_data(X, y)
print(f"Train: {data['train']['X'].shape}, Val: {data['val']['X'].shape}, Test: {data['test']['X'].shape}")

## 2. Build Baseline Model

In [ ]:
from src.model import Model
from src.layers import Linear
from src.activations import ReLU, Sigmoid, Tanh
from src.losses import BCELoss
from src.optimizers import Adam

# Build baseline model
model = Model()
model.add_module(Linear(10, 32), 'linear1')
model.add_module(ReLU(), 'relu1')
model.add_module(Linear(32, 16), 'linear2')
model.add_module(ReLU(), 'relu2')
model.add_module(Linear(16, 1), 'linear3')
model.add_module(Sigmoid(), 'sigmoid')

print_model_summary(model)

## 3. Train Baseline Model

In [ ]:
from src.trainer import Trainer
from src.visualization import plot_training_history

# Setup
loss_fn = BCELoss()
optimizer = Adam(lr=0.001)
trainer = Trainer(model, loss_fn, optimizer, batch_size=32)

# Train
history = trainer.fit(
    data['train']['X'], data['train']['y'],
    data['val']['X'], data['val']['y'],
    epochs=100,
    early_stopping=True,
    patience=15
)

# Plot
plot_training_history(history)
print("\n✓ Training complete!")

## 4. Evaluate on Test Set

In [ ]:
# Evaluate
_, test_metrics = trainer.evaluate(data['test']['X'], data['test']['y'])

print("\nTest Set Metrics:")
print("="*40)
for metric, value in test_metrics.items():
    print(f"{metric:15s}: {value:.4f}")
print("="*40)

## 5. Visualizations

In [ ]:
from src.visualization import plot_confusion_matrix, plot_roc_curve

# Predictions
y_test = data['test']['y']
y_proba = trainer.predict_proba(data['test']['X'])
y_pred = trainer.predict(data['test']['X'])

# Confusion matrix
plot_confusion_matrix(y_test, y_pred)

# ROC curve
plot_roc_curve(y_test, y_proba)